# 04 — Feature Engineering

## Sales Order, Backlog & Inventory Analytics

### Objective

This notebook transforms the cleaned datasets into analysis-ready datasets by creating reusable business and analytical features.

Feature engineering will focus on variables useful for:

- Sales and order analysis
- Backlog analysis
- Fulfillment performance
- Inventory analysis
- Shipment and delivery performance
- Supplier and procurement performance
- Time-based analysis
- SQL analysis and Power BI reporting

Only analytically meaningful features will be created.

Exploratory analysis, visualization, statistical analysis, and business conclusions are intentionally excluded from this notebook.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

print("\nFeature Engineering environment initialized successfully.")

Pandas version : 3.0.6
NumPy version  : 2.5.3

Feature Engineering environment initialized successfully.


## Step 2 — Define Processed Data Path

Define the project and processed-data directories.

Feature engineering will use only the validated cleaned datasets generated by `03_Data_Cleaning.ipynb`.

The raw datasets will not be used in this notebook.

In [2]:
# Define project paths

project_path = Path.cwd().parent

processed_data_path = (
    project_path / "data" / "processed"
)

print("Project Path        :", project_path)
print("Processed Data Path :", processed_data_path)

print(
    "\nProcessed Data Folder Exists :",
    processed_data_path.exists()
)

Project Path        : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics
Processed Data Path : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics\data\processed

Processed Data Folder Exists : True


## Step 3 — Verify Cleaned Dataset Files

Verify that all cleaned datasets generated during the Data Cleaning phase are available in the processed-data directory before beginning feature engineering.

In [3]:
# Check cleaned CSV files

clean_files = sorted(
    processed_data_path.glob("*_clean.csv")
)

print(
    f"Cleaned CSV Files Found : "
    f"{len(clean_files)}\n"
)

for file in clean_files:
    print(file.name)

Cleaned CSV Files Found : 10

customers_clean.csv
inventory_clean.csv
inventory_transactions_clean.csv
products_clean.csv
purchase_orders_clean.csv
sales_order_items_clean.csv
sales_orders_clean.csv
shipments_clean.csv
suppliers_clean.csv
warehouses_clean.csv


## Step 4 — Load Cleaned Datasets

Load all validated cleaned datasets from the processed-data directory.

Date columns are parsed during loading so that the datetime types established during data cleaning are restored for feature engineering.

In [4]:
# Load cleaned datasets

customers = pd.read_csv(
    processed_data_path / "customers_clean.csv",
    parse_dates=["customer_since"]
)

products = pd.read_csv(
    processed_data_path / "products_clean.csv"
)

sales_orders = pd.read_csv(
    processed_data_path / "sales_orders_clean.csv",
    parse_dates=[
        "order_date",
        "requested_delivery_date"
    ]
)

sales_order_items = pd.read_csv(
    processed_data_path / "sales_order_items_clean.csv",
    parse_dates=["order_date"]
)

inventory = pd.read_csv(
    processed_data_path / "inventory_clean.csv"
)

inventory_transactions = pd.read_csv(
    processed_data_path / "inventory_transactions_clean.csv",
    parse_dates=["transaction_date"]
)

warehouses = pd.read_csv(
    processed_data_path / "warehouses_clean.csv"
)

shipments = pd.read_csv(
    processed_data_path / "shipments_clean.csv",
    parse_dates=[
        "shipment_date",
        "requested_delivery_date",
        "actual_delivery_date"
    ]
)

suppliers = pd.read_csv(
    processed_data_path / "suppliers_clean.csv"
)

purchase_orders = pd.read_csv(
    processed_data_path / "purchase_orders_clean.csv",
    parse_dates=[
        "po_date",
        "expected_date",
        "actual_receipt_date"
    ]
)


print("Cleaned datasets loaded successfully.\n")

datasets = {
    "Customers": customers,
    "Products": products,
    "Sales Orders": sales_orders,
    "Sales Order Items": sales_order_items,
    "Inventory": inventory,
    "Inventory Transactions": inventory_transactions,
    "Warehouses": warehouses,
    "Shipments": shipments,
    "Suppliers": suppliers,
    "Purchase Orders": purchase_orders
}

for name, df in datasets.items():
    print(
        f"{name:<24} "
        f"{df.shape[0]:>8,} rows × "
        f"{df.shape[1]:>2} columns"
    )

Cleaned datasets loaded successfully.

Customers                   2,500 rows ×  6 columns
Products                      500 rows × 10 columns
Sales Orders               25,000 rows × 14 columns
Sales Order Items          61,397 rows × 16 columns
Inventory                   3,000 rows × 11 columns
Inventory Transactions    110,000 rows ×  6 columns
Warehouses                      6 rows ×  5 columns
Shipments                  24,798 rows × 14 columns
Suppliers                     100 rows ×  7 columns
Purchase Orders             8,500 rows × 14 columns


## Step 5 — Sales Order Time Features

Create reusable calendar attributes from `order_date` for time-based sales and backlog analysis.

These features support:

- Monthly and yearly sales trends
- Quarterly performance analysis
- Month-based comparisons
- Weekday ordering patterns
- SQL aggregation and Power BI reporting

The original `order_date` field is preserved.

In [5]:
# Create Sales Order time features

sales_orders["order_year"] = (
    sales_orders["order_date"].dt.year
)

sales_orders["order_quarter"] = (
    "Q" + sales_orders["order_date"]
    .dt.quarter.astype(str)
)

sales_orders["order_month_no"] = (
    sales_orders["order_date"].dt.month
)

sales_orders["order_month"] = (
    sales_orders["order_date"].dt.month_name()
)

sales_orders["order_year_month"] = (
    sales_orders["order_date"]
    .dt.to_period("M")
    .astype(str)
)

sales_orders["order_day_of_week"] = (
    sales_orders["order_date"].dt.day_name()
)


print("Sales Order Time Features Created.\n")

print(
    sales_orders[
        [
            "order_id",
            "order_date",
            "order_year",
            "order_quarter",
            "order_month_no",
            "order_month",
            "order_year_month",
            "order_day_of_week"
        ]
    ]
    .head(10)
    .to_string(index=False)
)


print(
    f"\nSales Orders Column Count : "
    f"{sales_orders.shape[1]}"
)

Sales Order Time Features Created.

order_id order_date  order_year order_quarter  order_month_no order_month order_year_month order_day_of_week
SO000001 2024-12-16        2024            Q4              12    December          2024-12            Monday
SO000002 2026-09-21        2026            Q3               9   September          2026-09            Monday
SO000003 2025-04-13        2025            Q2               4       April          2025-04            Sunday
SO000004 2025-11-13        2025            Q4              11    November          2025-11          Thursday
SO000005 2026-07-06        2026            Q3               7        July          2026-07            Monday
SO000006 2025-10-02        2025            Q4              10     October          2025-10          Thursday
SO000007 2024-09-27        2024            Q3               9   September          2024-09            Friday
SO000008 2025-09-12        2025            Q3               9   September          2025-09  

## Step 6 — Requested Delivery Lead Time

Calculate the number of days between the order date and the requested delivery date.

This feature represents the delivery window requested by the customer and can later support fulfillment and delivery-performance analysis.

The original date fields are preserved.

In [6]:
# Calculate requested delivery lead time

sales_orders["requested_lead_time_days"] = (
    sales_orders["requested_delivery_date"]
    - sales_orders["order_date"]
).dt.days


print("Requested Lead Time Feature Created.\n")

print(
    sales_orders[
        [
            "order_id",
            "order_date",
            "requested_delivery_date",
            "requested_lead_time_days"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    "\nMissing Values :",
    sales_orders["requested_lead_time_days"].isna().sum()
)

print(
    "Negative Lead Times :",
    (sales_orders["requested_lead_time_days"] < 0).sum()
)

print(
    "Sales Orders Column Count :",
    sales_orders.shape[1]
)

Requested Lead Time Feature Created.

order_id order_date requested_delivery_date  requested_lead_time_days
SO000001 2024-12-16              2024-12-25                         9
SO000002 2026-09-21              2026-10-03                        12
SO000003 2025-04-13              2025-04-17                         4
SO000004 2025-11-13              2025-11-17                         4
SO000005 2026-07-06              2026-07-09                         3
SO000006 2025-10-02              2025-10-13                        11
SO000007 2024-09-27              2024-10-08                        11
SO000008 2025-09-12              2025-09-14                         2
SO000009 2025-02-02              2025-02-09                         7
SO000010 2025-02-27              2025-03-08                         9

Missing Values : 0
Negative Lead Times : 0
Sales Orders Column Count : 21


## Step 7 — Order Fulfillment & Backlog Features

Create analytical flags that identify:

- Orders that are fully fulfilled
- Orders that currently contain backlog

These features simplify later SQL aggregation, backlog analysis, KPI calculations, and Power BI reporting.

In [7]:
# Create fulfillment and backlog flags

sales_orders["fully_fulfilled_flag"] = np.where(
    sales_orders["fulfilled_qty"] >= sales_orders["ordered_qty"],
    "Y",
    "N"
)

sales_orders["has_backlog_flag"] = np.where(
    sales_orders["backlog_qty"] > 0,
    "Y",
    "N"
)


print("Order Fulfillment & Backlog Features Created.\n")

print(
    sales_orders[
        [
            "order_id",
            "ordered_qty",
            "fulfilled_qty",
            "backlog_qty",
            "fully_fulfilled_flag",
            "has_backlog_flag"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nFully Fulfilled Flag:")
print(
    sales_orders["fully_fulfilled_flag"]
    .value_counts(dropna=False)
)

print("\nHas Backlog Flag:")
print(
    sales_orders["has_backlog_flag"]
    .value_counts(dropna=False)
)

print(
    "\nSales Orders Column Count :",
    sales_orders.shape[1]
)

Order Fulfillment & Backlog Features Created.

order_id  ordered_qty  fulfilled_qty  backlog_qty fully_fulfilled_flag has_backlog_flag
SO000001           10             10            0                    Y                N
SO000002           10             10            0                    Y                N
SO000003           13             13            0                    Y                N
SO000004           12              5            7                    N                Y
SO000005            8              6            2                    N                Y
SO000006           17             17            0                    Y                N
SO000007           12              7            5                    N                Y
SO000008           18             18            0                    Y                N
SO000009           42             42            0                    Y                N
SO000010           13             13            0                    Y   

## Step 8 — Sales Order Backlog Percentage

Calculate the percentage of ordered quantity that remains in backlog.

This feature provides a standardized measure of backlog severity, allowing orders of different sizes to be compared consistently.

A value of:

- `0%` indicates no backlog.
- Higher percentages indicate greater unfulfilled order quantities.

In [8]:
# Create backlog percentage

sales_orders["backlog_pct"] = (
    sales_orders["backlog_qty"]
    / sales_orders["ordered_qty"]
) * 100


print("Sales Order Backlog Percentage Created.\n")

print(
    sales_orders[
        [
            "order_id",
            "ordered_qty",
            "fulfilled_qty",
            "backlog_qty",
            "backlog_pct"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    "\nMissing Values :",
    sales_orders["backlog_pct"].isna().sum()
)

print(
    "Minimum Backlog % :",
    round(sales_orders["backlog_pct"].min(), 2)
)

print(
    "Maximum Backlog % :",
    round(sales_orders["backlog_pct"].max(), 2)
)

print(
    "Sales Orders Column Count :",
    sales_orders.shape[1]
)

Sales Order Backlog Percentage Created.

order_id  ordered_qty  fulfilled_qty  backlog_qty  backlog_pct
SO000001           10             10            0     0.000000
SO000002           10             10            0     0.000000
SO000003           13             13            0     0.000000
SO000004           12              5            7    58.333333
SO000005            8              6            2    25.000000
SO000006           17             17            0     0.000000
SO000007           12              7            5    41.666667
SO000008           18             18            0     0.000000
SO000009           42             42            0     0.000000
SO000010           13             13            0     0.000000

Missing Values : 0
Minimum Backlog % : 0.0
Maximum Backlog % : 100.0
Sales Orders Column Count : 24


## Step 9 — Sales Order Value Fulfillment Features

Create monetary fulfillment features to complement quantity-based fulfillment metrics.

The following features are created:

- `fulfilled_value` — portion of order value that has been fulfilled.
- `fulfilled_value_pct` — percentage of total order value that has been fulfilled.

These features support revenue fulfillment and backlog-value analysis.

In [9]:
# Create value-based fulfillment features

sales_orders["fulfilled_value"] = (
    sales_orders["order_value"]
    - sales_orders["backlog_value"]
)

sales_orders["fulfilled_value_pct"] = (
    sales_orders["fulfilled_value"]
    / sales_orders["order_value"]
) * 100


print("Sales Order Value Fulfillment Features Created.\n")

print(
    sales_orders[
        [
            "order_id",
            "order_value",
            "backlog_value",
            "fulfilled_value",
            "fulfilled_value_pct"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    "\nMissing Fulfilled Value % :",
    sales_orders["fulfilled_value_pct"].isna().sum()
)

print(
    "Minimum Fulfilled Value % :",
    round(sales_orders["fulfilled_value_pct"].min(), 2)
)

print(
    "Maximum Fulfilled Value % :",
    round(sales_orders["fulfilled_value_pct"].max(), 2)
)

print(
    "Sales Orders Column Count :",
    sales_orders.shape[1]
)

Sales Order Value Fulfillment Features Created.

order_id  order_value  backlog_value  fulfilled_value  fulfilled_value_pct
SO000001     17798.54           0.00         17798.54           100.000000
SO000002     22603.12           0.00         22603.12           100.000000
SO000003     15553.32           0.00         15553.32           100.000000
SO000004     91258.31       58725.59         32532.72            35.649049
SO000005     11517.72        3319.40          8198.32            71.180060
SO000006    136849.66           0.00        136849.66           100.000000
SO000007     73609.05       26978.62         46630.43            63.348773
SO000008     96918.46           0.00         96918.46           100.000000
SO000009    142597.86           0.00        142597.86           100.000000
SO000010     45605.56           0.00         45605.56           100.000000

Missing Fulfilled Value % : 0
Minimum Fulfilled Value % : 0.0
Maximum Fulfilled Value % : 100.0
Sales Orders Column Count : 2

## Step 10 — Add Product Cost to Sales Order Items

Enrich sales order line items with product unit cost from the product master.

The product cost is required to calculate line-level cost, gross profit, and profit margin in subsequent feature-engineering steps.

The merge uses `product_id` as the relationship key.

In [10]:
# Add product unit cost to sales order items

sales_order_items = sales_order_items.merge(
    products[
        [
            "product_id",
            "unit_cost"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)


print("Product Cost Added to Sales Order Items.\n")

print(
    sales_order_items[
        [
            "order_item_id",
            "product_id",
            "unit_price",
            "discount_pct",
            "unit_cost"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    "\nMissing Unit Cost :",
    sales_order_items["unit_cost"].isna().sum()
)

print(
    "Sales Order Items Row Count :",
    sales_order_items.shape[0]
)

print(
    "Sales Order Items Column Count :",
    sales_order_items.shape[1]
)

Product Cost Added to Sales Order Items.

order_item_id product_id  unit_price  discount_pct  unit_cost
   SOI0000001    SKU0200     1999.29          0.00    1375.99
   SOI0000002    SKU0259     1719.54          0.05    1463.48
   SOI0000003    SKU0192      507.52          0.08     395.34
   SOI0000004    SKU0470     6667.21          0.15    5406.00
   SOI0000005    SKU0193     1352.92          0.08    1066.12
   SOI0000006    SKU0118     1165.20          0.15     976.38
   SOI0000007    SKU0119     1690.27          0.15    1290.01
   SOI0000008    SKU0362     6639.33          0.02    5060.58
   SOI0000009    SKU0021     8389.37          0.00    6313.08
   SOI0000010    SKU0325      980.03          0.05     777.58

Missing Unit Cost : 0
Sales Order Items Row Count : 61397
Sales Order Items Column Count : 17


## Step 11 — Line-Level Cost & Profit Features

Create line-level profitability features using ordered quantity, product unit cost, and net order value.

The following features are created:

- `line_cost` — total product cost for the ordered quantity.
- `gross_profit` — net order value minus line cost.
- `profit_margin_pct` — gross profit as a percentage of net order value.

These features will support later product, category, customer, and profitability analysis.

In [11]:
# Create line-level profitability features

sales_order_items["line_cost"] = (
    sales_order_items["ordered_qty"]
    * sales_order_items["unit_cost"]
)

sales_order_items["gross_profit"] = (
    sales_order_items["net_order_value"]
    - sales_order_items["line_cost"]
)

sales_order_items["profit_margin_pct"] = (
    sales_order_items["gross_profit"]
    / sales_order_items["net_order_value"]
) * 100


print("Line-Level Profitability Features Created.\n")

print(
    sales_order_items[
        [
            "order_item_id",
            "ordered_qty",
            "net_order_value",
            "unit_cost",
            "line_cost",
            "gross_profit",
            "profit_margin_pct"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    "\nMissing Profit Margin :",
    sales_order_items["profit_margin_pct"].isna().sum()
)

print(
    "Loss-Making Lines :",
    (sales_order_items["gross_profit"] < 0).sum()
)

print(
    "Sales Order Items Column Count :",
    sales_order_items.shape[1]
)

Line-Level Profitability Features Created.

order_item_id  ordered_qty  net_order_value  unit_cost  line_cost  gross_profit  profit_margin_pct
   SOI0000001            4          7997.16    1375.99    5503.96       2493.20          31.176068
   SOI0000002            6          9801.38    1463.48    8780.88       1020.50          10.411799
   SOI0000003            4          1867.67     395.34    1581.36        286.31          15.329796
   SOI0000004            3         17001.39    5406.00   16218.00        783.39           4.607800
   SOI0000005            3          3734.06    1066.12    3198.36        535.70          14.346315
   SOI0000006            7          6932.94     976.38    6834.66         98.28           1.417580
   SOI0000007            6          8620.38    1290.01    7740.06        880.32          10.212079
   SOI0000008            5         32532.72    5060.58   25302.90       7229.82          22.223226
   SOI0000009            7         58725.59    6313.08   44191.56

## Step 12 — Inventory Coverage Features

Create inventory coverage metrics using available inventory and average daily demand.

The following feature is created:

- `inventory_days_on_hand` — estimated number of days the available inventory can support based on average daily demand.

This feature will support inventory sufficiency, replenishment, and stock-risk analysis.

In [12]:
# Create inventory coverage feature

inventory["inventory_days_on_hand"] = (
    inventory["available_qty"]
    / inventory["avg_daily_demand"]
)


print("Inventory Coverage Feature Created.\n")

print(
    inventory[
        [
            "product_id",
            "warehouse_id",
            "available_qty",
            "avg_daily_demand",
            "inventory_days_on_hand"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    "\nMissing Inventory Days on Hand :",
    inventory["inventory_days_on_hand"].isna().sum()
)

print(
    "Negative Inventory Days on Hand :",
    (inventory["inventory_days_on_hand"] < 0).sum()
)

print(
    "Inventory Column Count :",
    inventory.shape[1]
)

Inventory Coverage Feature Created.

product_id warehouse_id  available_qty  avg_daily_demand  inventory_days_on_hand
   SKU0001         WH01              0              0.33                0.000000
   SKU0001         WH02              3              0.33                9.090909
   SKU0001         WH03             22              0.33               66.666667
   SKU0001         WH04              5              0.33               15.151515
   SKU0001         WH05              3              0.33                9.090909
   SKU0001         WH06              8              0.33               24.242424
   SKU0002         WH01              0              0.12                0.000000
   SKU0002         WH02              8              0.12               66.666667
   SKU0002         WH03              3              0.12               25.000000
   SKU0002         WH04              3              0.12               25.000000

Missing Inventory Days on Hand : 0
Negative Inventory Days on Hand : 0


## Step 13 — Inventory Reorder Risk Feature

Create a replenishment indicator by comparing available inventory with the defined reorder point.

The following feature is created:

- `reorder_required_flag` — identifies product-warehouse combinations where available inventory is at or below the reorder point.

This feature supports replenishment prioritization and inventory-risk analysis.

In [13]:
# Create inventory reorder requirement flag

inventory["reorder_required_flag"] = np.where(
    inventory["available_qty"] <= inventory["reorder_point"],
    "Y",
    "N"
)


print("Inventory Reorder Risk Feature Created.\n")

print(
    inventory[
        [
            "product_id",
            "warehouse_id",
            "available_qty",
            "reorder_point",
            "reorder_required_flag"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nReorder Required Flag:")
print(
    inventory["reorder_required_flag"]
    .value_counts(dropna=False)
)

print(
    "\nInventory Column Count :",
    inventory.shape[1]
)

Inventory Reorder Risk Feature Created.

product_id warehouse_id  available_qty  reorder_point reorder_required_flag
   SKU0001         WH01              0              6                     Y
   SKU0001         WH02              3              9                     Y
   SKU0001         WH03             22             10                     N
   SKU0001         WH04              5              7                     Y
   SKU0001         WH05              3              8                     Y
   SKU0001         WH06              8              7                     N
   SKU0002         WH01              0              4                     Y
   SKU0002         WH02              8              5                     N
   SKU0002         WH03              3              3                     Y
   SKU0002         WH04              3              5                     Y

Reorder Required Flag:
reorder_required_flag
N    1689
Y    1311
Name: count, dtype: int64

Inventory Column Count : 13


## Step 14 — Inventory Allocation Features

Create inventory allocation metrics to measure how much on-hand inventory is already committed to customer demand.

The following features are created:

- `allocation_rate_pct` — percentage of on-hand inventory currently allocated.
- `unallocated_qty` — inventory remaining after allocated quantity is deducted.

These features support inventory utilization and availability analysis.

In [14]:
# Create inventory allocation features

inventory["unallocated_qty"] = (
    inventory["on_hand_qty"]
    - inventory["allocated_qty"]
)

inventory["allocation_rate_pct"] = np.where(
    inventory["on_hand_qty"] > 0,
    (
        inventory["allocated_qty"]
        / inventory["on_hand_qty"]
    ) * 100,
    0
)


print("Inventory Allocation Features Created.\n")

print(
    inventory[
        [
            "product_id",
            "warehouse_id",
            "on_hand_qty",
            "allocated_qty",
            "unallocated_qty",
            "allocation_rate_pct"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    "\nMissing Allocation Rate :",
    inventory["allocation_rate_pct"].isna().sum()
)

print(
    "Negative Unallocated Quantity :",
    (inventory["unallocated_qty"] < 0).sum()
)

print(
    "Inventory Column Count :",
    inventory.shape[1]
)

Inventory Allocation Features Created.

product_id warehouse_id  on_hand_qty  allocated_qty  unallocated_qty  allocation_rate_pct
   SKU0001         WH01            0              0                0             0.000000
   SKU0001         WH02            5              2                3            40.000000
   SKU0001         WH03           24              2               22             8.333333
   SKU0001         WH04            5              0                5             0.000000
   SKU0001         WH05            4              1                3            25.000000
   SKU0001         WH06            9              1                8            11.111111
   SKU0002         WH01            0              0                0             0.000000
   SKU0002         WH02            9              1                8            11.111111
   SKU0002         WH03            4              1                3            25.000000
   SKU0002         WH04            3              0         

## Step 15 — Shipment Time Features

Create reusable calendar attributes from `shipment_date` for delivery and logistics analysis.

The following features are created:

- `shipment_year`
- `shipment_quarter`
- `shipment_month_no`
- `shipment_month`
- `shipment_year_month`

The existing chronology-safe delivery KPI fields are preserved and will not be recalculated from invalid shipment records.

In [15]:
# Create shipment time features

shipments["shipment_year"] = (
    shipments["shipment_date"].dt.year
)

shipments["shipment_quarter"] = (
    "Q" + shipments["shipment_date"]
    .dt.quarter.astype(str)
)

shipments["shipment_month_no"] = (
    shipments["shipment_date"].dt.month
)

shipments["shipment_month"] = (
    shipments["shipment_date"].dt.month_name()
)

shipments["shipment_year_month"] = (
    shipments["shipment_date"]
    .dt.to_period("M")
    .astype(str)
)


print("Shipment Time Features Created.\n")

print(
    shipments[
        [
            "shipment_id",
            "shipment_date",
            "shipment_year",
            "shipment_quarter",
            "shipment_month_no",
            "shipment_month",
            "shipment_year_month",
            "chronology_valid_flag"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    "\nShipment Column Count :",
    shipments.shape[1]
)

Shipment Time Features Created.

shipment_id shipment_date  shipment_year shipment_quarter  shipment_month_no shipment_month shipment_year_month chronology_valid_flag
  SHP000001    2024-12-21           2024               Q4                 12       December             2024-12                     Y
  SHP000002    2026-09-23           2026               Q3                  9      September             2026-09                     Y
  SHP000003    2025-04-15           2025               Q2                  4          April             2025-04                     Y
  SHP000004    2025-11-18           2025               Q4                 11       November             2025-11                     N
  SHP000005    2026-07-11           2026               Q3                  7           July             2026-07                     N
  SHP000006    2025-10-03           2025               Q4                 10        October             2025-10                     Y
  SHP000007    2024-09-29    

## Step 16 — Shipment Delivery Performance Features

Create analysis-friendly delivery performance categories using the chronology-safe shipment KPI fields.

The following features are created:

- `delivery_status` — classifies shipments as On Time, Late, or Invalid Chronology.
- `delay_severity` — groups valid shipments according to the magnitude of delivery delay.

Records with invalid shipment chronology remain explicitly identified and are not treated as valid delivery-performance observations.

In [16]:
# Create shipment delivery performance features

shipments["delivery_status"] = np.select(
    [
        shipments["chronology_valid_flag"].eq("N"),
        shipments["on_time_flag_clean"].eq(1),
        shipments["on_time_flag_clean"].eq(0)
    ],
    [
        "Invalid Chronology",
        "On Time",
        "Late"
    ],
    default="Unknown"
)


shipments["delay_severity"] = np.select(
    [
        shipments["chronology_valid_flag"].eq("N"),
        shipments["delivery_delay_days_clean"] <= 0,
        shipments["delivery_delay_days_clean"].between(1, 2),
        shipments["delivery_delay_days_clean"].between(3, 5),
        shipments["delivery_delay_days_clean"] >= 6
    ],
    [
        "Invalid Chronology",
        "On Time",
        "1-2 Days Late",
        "3-5 Days Late",
        "6+ Days Late"
    ],
    default="Unknown"
)


print("Shipment Delivery Performance Features Created.\n")

print("Delivery Status:")
print(
    shipments["delivery_status"]
    .value_counts(dropna=False)
)

print("\nDelay Severity:")
print(
    shipments["delay_severity"]
    .value_counts(dropna=False)
)

print(
    "\nUnknown Delivery Status :",
    (shipments["delivery_status"] == "Unknown").sum()
)

print(
    "Unknown Delay Severity :",
    (shipments["delay_severity"] == "Unknown").sum()
)

print(
    "Shipment Column Count :",
    shipments.shape[1]
)

Shipment Delivery Performance Features Created.

Delivery Status:
delivery_status
Unknown               20331
Invalid Chronology     4467
Name: count, dtype: int64

Delay Severity:
delay_severity
On Time               14480
Invalid Chronology     4467
6+ Days Late           2961
3-5 Days Late          1826
1-2 Days Late          1064
Name: count, dtype: int64

Unknown Delivery Status : 20331
Unknown Delay Severity : 0
Shipment Column Count : 21


In [17]:
# Correct delivery status using chronology-safe delay values

shipments["delivery_status"] = np.select(
    [
        shipments["chronology_valid_flag"].eq("N"),
        shipments["delivery_delay_days_clean"] <= 0,
        shipments["delivery_delay_days_clean"] > 0
    ],
    [
        "Invalid Chronology",
        "On Time",
        "Late"
    ],
    default="Unknown"
)


print("Corrected Delivery Status:\n")

print(
    shipments["delivery_status"]
    .value_counts(dropna=False)
)

print(
    "\nUnknown Delivery Status :",
    (shipments["delivery_status"] == "Unknown").sum()
)

print(
    "Total Shipments :",
    len(shipments)
)

Corrected Delivery Status:

delivery_status
On Time               14480
Late                   5851
Invalid Chronology     4467
Name: count, dtype: int64

Unknown Delivery Status : 0
Total Shipments : 24798


## Step 17 — Purchase Order Time & Lead-Time Features

Create procurement time features from purchase order dates.

The following features are created:

- `po_year`
- `po_quarter`
- `po_month_no`
- `po_month`
- `po_year_month`
- `planned_lead_time_days` — number of days between PO creation and expected receipt.
- `actual_lead_time_days` — number of days between PO creation and actual receipt.

Open purchase orders without an actual receipt date will retain missing actual lead time values.

In [18]:
# Create purchase order time and lead-time features

purchase_orders["po_year"] = (
    purchase_orders["po_date"].dt.year
)

purchase_orders["po_quarter"] = (
    "Q" + purchase_orders["po_date"]
    .dt.quarter.astype(str)
)

purchase_orders["po_month_no"] = (
    purchase_orders["po_date"].dt.month
)

purchase_orders["po_month"] = (
    purchase_orders["po_date"].dt.month_name()
)

purchase_orders["po_year_month"] = (
    purchase_orders["po_date"]
    .dt.to_period("M")
    .astype(str)
)

purchase_orders["planned_lead_time_days"] = (
    purchase_orders["expected_date"]
    - purchase_orders["po_date"]
).dt.days

purchase_orders["actual_lead_time_days"] = (
    purchase_orders["actual_receipt_date"]
    - purchase_orders["po_date"]
).dt.days


print("Purchase Order Time & Lead-Time Features Created.\n")

print(
    purchase_orders[
        [
            "po_id",
            "po_date",
            "expected_date",
            "actual_receipt_date",
            "planned_lead_time_days",
            "actual_lead_time_days"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print(
    "\nMissing Planned Lead Time :",
    purchase_orders["planned_lead_time_days"].isna().sum()
)

print(
    "Missing Actual Lead Time :",
    purchase_orders["actual_lead_time_days"].isna().sum()
)

print(
    "Purchase Orders Column Count :",
    purchase_orders.shape[1]
)

Purchase Order Time & Lead-Time Features Created.

   po_id    po_date expected_date actual_receipt_date  planned_lead_time_days  actual_lead_time_days
PO000001 2025-12-03    2025-12-21          2025-12-25                      18                   22.0
PO000002 2024-04-24    2024-05-05          2024-05-04                      11                   10.0
PO000003 2026-03-29    2026-04-09          2026-04-07                      11                    9.0
PO000004 2026-07-20    2026-08-01          2026-08-09                      12                   20.0
PO000005 2025-09-09    2025-09-20          2025-09-18                      11                    9.0
PO000006 2026-05-13    2026-05-28          2026-05-26                      15                   13.0
PO000007 2024-12-05    2024-12-13          2024-12-28                       8                   23.0
PO000008 2026-01-20    2026-01-29          2026-02-09                       9                   20.0
PO000009 2024-04-29    2024-05-10       

## Step 18 — Purchase Order Delivery Performance Features

Create procurement delivery-performance categories using supplier delay days.

The following features are created:

- `supplier_delivery_status` — classifies purchase orders as Open, Early, On Time, or Late.
- `supplier_delay_severity` — groups received purchase orders according to supplier delay magnitude.

Open purchase orders are kept as a separate category because they do not yet have an actual receipt date or supplier delay value.

## Step 18 — Purchase Order Delivery Performance Features

Create procurement delivery-performance categories using supplier delay days.

The following features are created:

- `supplier_delivery_status` — classifies purchase orders as Open, Early, On Time, or Late.
- `supplier_delay_severity` — groups received purchase orders according to supplier delay magnitude.

Open purchase orders are kept as a separate category because they do not yet have an actual receipt date or supplier delay value.

In [21]:
# Create supplier delivery performance features

purchase_orders["supplier_delivery_status"] = np.select(
    [
        purchase_orders["po_status"].eq("Open"),
        purchase_orders["supplier_delay_days"] < 0,
        purchase_orders["supplier_delay_days"] == 0,
        purchase_orders["supplier_delay_days"] > 0
    ],
    [
        "Open",
        "Early",
        "On Time",
        "Late"
    ],
    default="Unknown"
)

purchase_orders["supplier_delay_severity"] = np.select(
    [
        purchase_orders["po_status"].eq("Open"),
        purchase_orders["supplier_delay_days"] < 0,
        purchase_orders["supplier_delay_days"] == 0,
        purchase_orders["supplier_delay_days"].between(1, 3),
        purchase_orders["supplier_delay_days"].between(4, 7),
        purchase_orders["supplier_delay_days"] >= 8
    ],
    [
        "Open",
        "Early",
        "On Time",
        "1-3 Days Late",
        "4-7 Days Late",
        "8+ Days Late"
    ],
    default="Unknown"
)

print("Supplier Delivery Performance Features Created.\n")

print("Supplier Delivery Status:")
print(
    purchase_orders["supplier_delivery_status"]
    .value_counts(dropna=False)
)

print("\nSupplier Delay Severity:")
print(
    purchase_orders["supplier_delay_severity"]
    .value_counts(dropna=False)
)

print(
    "\nUnknown Delivery Status :",
    (purchase_orders["supplier_delivery_status"] == "Unknown").sum()
)

print(
    "Unknown Delay Severity :",
    (purchase_orders["supplier_delay_severity"] == "Unknown").sum()
)

print(
    "Purchase Orders Column Count :",
    purchase_orders.shape[1]
)

Supplier Delivery Performance Features Created.

Supplier Delivery Status:
supplier_delivery_status
Early      3759
Late       2935
On Time    1776
Open         30
Name: count, dtype: int64

Supplier Delay Severity:
supplier_delay_severity
Early            3759
1-3 Days Late    1954
On Time          1776
8+ Days Late      718
4-7 Days Late     263
Open               30
Name: count, dtype: int64

Unknown Delivery Status : 0
Unknown Delay Severity : 0
Purchase Orders Column Count : 23


## Step 19 — Final Feature Engineering Validation

Perform final validation of the feature-engineered datasets before export.

The validation confirms:

- Original row counts have been preserved.
- Key engineered features contain no unexpected missing values.
- Shipment chronology exclusions remain intact.
- Open purchase orders retain their intentional missing actual lead-time values.
- No unexpected analytical categories were introduced.

In [22]:
# Final feature engineering validation

expected_rows = {
    "Customers": 2500,
    "Products": 500,
    "Sales Orders": 25000,
    "Sales Order Items": 61397,
    "Inventory": 3000,
    "Inventory Transactions": 110000,
    "Warehouses": 6,
    "Shipments": 24798,
    "Suppliers": 100,
    "Purchase Orders": 8500
}

feature_datasets = {
    "Customers": customers,
    "Products": products,
    "Sales Orders": sales_orders,
    "Sales Order Items": sales_order_items,
    "Inventory": inventory,
    "Inventory Transactions": inventory_transactions,
    "Warehouses": warehouses,
    "Shipments": shipments,
    "Suppliers": suppliers,
    "Purchase Orders": purchase_orders
}

print("ROW COUNT VALIDATION\n")

all_rows_valid = True

for name, df in feature_datasets.items():
    actual = len(df)
    expected = expected_rows[name]
    status = "PASS" if actual == expected else "FAIL"

    if actual != expected:
        all_rows_valid = False

    print(
        f"{name:<24} "
        f"Expected: {expected:>7,} | "
        f"Actual: {actual:>7,} | "
        f"{status}"
    )


print("\nCRITICAL FEATURE VALIDATION\n")

checks = {
    "Sales Order backlog_pct missing":
        sales_orders["backlog_pct"].isna().sum(),

    "Sales Order profit margin missing":
        sales_order_items["profit_margin_pct"].isna().sum(),

    "Inventory DOH missing":
        inventory["inventory_days_on_hand"].isna().sum(),

    "Shipment unknown delivery status":
        (shipments["delivery_status"] == "Unknown").sum(),

    "Shipment unknown delay severity":
        (shipments["delay_severity"] == "Unknown").sum(),

    "PO unknown delivery status":
        (purchase_orders["supplier_delivery_status"] == "Unknown").sum(),

    "PO unknown delay severity":
        (purchase_orders["supplier_delay_severity"] == "Unknown").sum()
}

for check, value in checks.items():
    print(f"{check:<40} : {value}")


print(
    "\nInvalid Shipment Chronology :",
    (shipments["chronology_valid_flag"] == "N").sum()
)

print(
    "Open PO Missing Actual Lead Time :",
    purchase_orders.loc[
        purchase_orders["po_status"] == "Open",
        "actual_lead_time_days"
    ].isna().sum()
)


critical_checks_pass = (
    all(value == 0 for value in checks.values())
    and
    (shipments["chronology_valid_flag"] == "N").sum() == 4467
    and
    purchase_orders.loc[
        purchase_orders["po_status"] == "Open",
        "actual_lead_time_days"
    ].isna().sum() == 30
)


print(
    "\nFINAL VALIDATION STATUS :",
    "PASSED"
    if all_rows_valid and critical_checks_pass
    else "REVIEW REQUIRED"
)

ROW COUNT VALIDATION

Customers                Expected:   2,500 | Actual:   2,500 | PASS
Products                 Expected:     500 | Actual:     500 | PASS
Sales Orders             Expected:  25,000 | Actual:  25,000 | PASS
Sales Order Items        Expected:  61,397 | Actual:  61,397 | PASS
Inventory                Expected:   3,000 | Actual:   3,000 | PASS
Inventory Transactions   Expected: 110,000 | Actual: 110,000 | PASS
Warehouses               Expected:       6 | Actual:       6 | PASS
Shipments                Expected:  24,798 | Actual:  24,798 | PASS
Suppliers                Expected:     100 | Actual:     100 | PASS
Purchase Orders          Expected:   8,500 | Actual:   8,500 | PASS

CRITICAL FEATURE VALIDATION

Sales Order backlog_pct missing          : 0
Sales Order profit margin missing        : 0
Inventory DOH missing                    : 0
Shipment unknown delivery status         : 0
Shipment unknown delay severity          : 0
PO unknown delivery status               : 

## Step 20 — Export Feature-Engineered Datasets

Export the final analysis-ready datasets containing the engineered features.

The feature-engineered datasets are stored separately from the cleaned datasets to preserve the output of the Data Cleaning phase.

These datasets will serve as the analysis-ready source for subsequent SQL, analytical, and Power BI workflows.

In [23]:
# Create feature-engineered output folder

featured_data_path = project_path / "data" / "featured"

featured_data_path.mkdir(
    parents=True,
    exist_ok=True
)


# Define datasets for export

export_datasets = {
    "customers_featured.csv": customers,
    "products_featured.csv": products,
    "sales_orders_featured.csv": sales_orders,
    "sales_order_items_featured.csv": sales_order_items,
    "inventory_featured.csv": inventory,
    "inventory_transactions_featured.csv": inventory_transactions,
    "warehouses_featured.csv": warehouses,
    "shipments_featured.csv": shipments,
    "suppliers_featured.csv": suppliers,
    "purchase_orders_featured.csv": purchase_orders
}


# Export datasets

for filename, df in export_datasets.items():

    df.to_csv(
        featured_data_path / filename,
        index=False,
        date_format="%Y-%m-%d"
    )


print("Feature-Engineered Datasets Exported.\n")

for filename, df in export_datasets.items():

    file_path = featured_data_path / filename

    print(
        f"{filename:<38} "
        f"{df.shape[0]:>7,} rows × "
        f"{df.shape[1]:>2} cols | "
        f"Exists: {file_path.exists()}"
    )


print(
    "\nExport Folder :",
    featured_data_path
)

print(
    "\nFEATURE ENGINEERING EXPORT STATUS :",
    "PASSED"
    if all(
        (featured_data_path / filename).exists()
        for filename in export_datasets
    )
    else "FAILED"
)

Feature-Engineered Datasets Exported.

customers_featured.csv                   2,500 rows ×  6 cols | Exists: True
products_featured.csv                      500 rows × 10 cols | Exists: True
sales_orders_featured.csv               25,000 rows × 26 cols | Exists: True
sales_order_items_featured.csv          61,397 rows × 20 cols | Exists: True
inventory_featured.csv                   3,000 rows × 15 cols | Exists: True
inventory_transactions_featured.csv    110,000 rows ×  6 cols | Exists: True
warehouses_featured.csv                      6 rows ×  5 cols | Exists: True
shipments_featured.csv                  24,798 rows × 21 cols | Exists: True
suppliers_featured.csv                     100 rows ×  7 cols | Exists: True
purchase_orders_featured.csv             8,500 rows × 23 cols | Exists: True

Export Folder : c:\Users\Admin\OneDrive\Documents\Sales_Order_Backlog_Inventory_Analytics\data\featured

FEATURE ENGINEERING EXPORT STATUS : PASSED
